The work of the bake-off is **building** the binders; comparing them is a much smaller step. This notebook builds all four over the annual-report store --- it generates training samples from the store, fine-tunes the compiler (arm B), indexes those samples for the frozen few-shot binder (arm C), and calibrates the patchable alias table (arm D) --- and only then reads the persisted crossover. The build cells need the GPU and the licensed backend; run them locally (execution is off at render time).

**Setup.** Load the store and the base LM through the installed package.

In [ ]:
from forgeloop import data_path
from forgeloop.rag import load_store
from knowlytix.knowledge.llm_backend import LocalTransformersBackend
from knowlytix.knowledge.rag.compiler import (build_compiler_dataset, train_compiler,
                                              CompilerSFTConfig, QWEN_4B)
store = load_store(str(data_path('gms_annual_report_store')))
llm = LocalTransformersBackend(QWEN_4B)   # rephraser for datagen + base for SFT

**Arm B, step 1 --- generate samples.** Enumerate the store's facts and pair each with a natural-language question over the DoE presentation factors; hold out a level and a fraction of facts for evaluation.

In [ ]:
splits = build_compiler_dataset(
    store, llm, group='comprehensive', variants_per_base=12,
    heldout_levels={'clarity': 'Misleading'}, heldout_fact_frac=0.15, seed=42)
print({k: len(v) for k, v in splits.items()})
print('one sample:', splits['train'][0])

**Arm B, step 2 --- fine-tune the SLM compiler** on the samples with a low-rank adapter, written into the store's `query_compiler/`.

In [ ]:
compiler_dir = train_compiler(
    splits['train'], out_dir=str(data_path('gms_annual_report_store', 'query_compiler')),
    config=CompilerSFTConfig(base_model=QWEN_4B, lora_r=16, lora_alpha=32, epochs=3))
print('compiler adapter:', compiler_dir)

**Arm C --- index the same samples** in the tuned encoder's space (no training); the k nearest are shown to a frozen model at inference.

In [ ]:
from knowlytix.embedding import FineTunedEmbedding
from knowlytix.knowledge.rag.bakeoff import ExemplarIndex
v_encoder = FineTunedEmbedding.load(str(data_path('gms_annual_report_store', 'tuned_encoder')))
index = ExemplarIndex.from_rows(splits['train'], v_encoder.encode)
print('exemplars indexed:', len(index))

**Arm D --- build a patchable alias table** for the head entity, with a calibrated encoder-nearest fallback; a mis-binding is fixed by one edit.

In [ ]:
from knowlytix.knowledge.rag.bakeoff import AliasTableResolver, calibrate_alias_resolver
from knowlytix.knowledge.rag.compiler.walk import StoreChainWalker
entities = StoreChainWalker(store).entities
tau = calibrate_alias_resolver(entities, v_encoder.encode, far_ceiling=0.05)['tau']
resolver = AliasTableResolver.from_store(store, encoder=v_encoder.encode, tau=tau)
resolver.add_alias('CP', 'cloud platform')   # a patch is one row, not a retrain
print('alias entries:', len(resolver.table), '| fallback tau:', round(tau, 3))

**The comparison, in one step.** With the binders built, the crossover and the G4 verdict are read from the persisted run.

In [ ]:
import json
report = json.load(open(data_path('enrichment', 'bakeoff_ABCD.json')))
decision = json.load(open(data_path('enrichment', 'bakeoff_decision.json')))
for name, e in report['arms'].items():
    o = e['overall']
    print(f"{name:12} acc={o['accuracy']:.3f} mis_bind={o['mis_bind_rate']:.3f} "
          f"holdout={e['holdout']['accuracy']:.3f} patch={e['patch_cost']}")
print('decision:', decision['recommended'], '| recused:', decision['recused'])